# FastDetect

### Utils.py

In [ ]:
import plotly.graph_objects as go
import os
from scipy.optimize import minimize
import math
from math import ceil, floor
import matplotlib.pyplot as plt
import pickle
from tqdm import tqdm

USE_GPU = os.getenv("USE_GPU", "0") == "1"

if USE_GPU:
    try:
        import cupy as xp
        import cupyx.scipy.fft as xfft
        on_gpu = True

        asnumpy = xp.asnumpy    # CuPy → NumPy
        asarray = xp.asarray   # Python/NumPy → CuPy
    except ImportError:
        # fallback if cupy not installed
        import numpy as xp
        import scipy.fft as xfft
        on_gpu = False

        asnumpy = lambda a: a
        asarray = xp.asarray
else:
    import numpy as xp
    import scipy.fft as xfft
    on_gpu = False

    asnumpy = lambda a: a
    asarray = xp.asarray

# -------- light helpers (backend-agnostic) --------
def wrap(x):
    return (x + xp.pi) % (2 * xp.pi) - xp.pi


def to_device(x):
    """Move/convert Python or NumPy data to the active backend array (CuPy on GPU, NumPy on CPU)."""
    return asarray(x)

def to_host(x):
    """Ensure a NumPy array on host memory (identity on CPU)."""
    return asnumpy(x)

def to_scalar(x):
    """Return a Python scalar if x is 0-d array/array-like; otherwise return x unchanged."""
    # Works for both NumPy and CuPy
    try:
        if getattr(x, "shape", None) == ():
            return x.item()
    except Exception:
        pass
    return x

def sqlist(lst):
    return xp.array([to_scalar(item) for item in lst])

def myfft(chirp_data, n, plan):
    if USE_GPU:
        return xfft.fftshift(xfft.fft(chirp_data.astype(xp.complex64), n=n, plan=plan))
    else:
        return xfft.fftshift(xfft.fft(chirp_data.astype(xp.complex64), n=n))


def optimize_1dfreq_fast(sig2, tsymbr, freq1, margin):
    def obj1(freq, xdata, ydata):
        return to_scalar(-xp.abs(ydata.dot(xp.exp(xdata * -1j * 2 * xp.pi * freq.item()))))
    result = minimize(obj1, to_scalar(freq1), args=(tsymbr, sig2), bounds=[(freq1 - margin, freq1 + margin)]) #!!!
    return result.x[0], - result.fun / xp.sum(xp.abs(sig2))


def around(x):
    """Round to nearest integer (works for Python num or 0-d array)."""
    return round(float(to_scalar(x)))

def optimize_1dfreq(sig2, tsymbr, freq, margin):
    def obj1(xdata, ydata, freq):
        return xp.abs(ydata.dot(xp.exp(-1j * 2 * xp.pi * freq * xdata)))

    margin = 500
    val = obj1(tsymbr, sig2, freq)
    for i in range(10):
        xvals = xp.linspace(freq - margin, freq + margin, 1001)
        yvals = [obj1(tsymbr, sig2, f) for f in xvals]
        yvals = sqlist(yvals)
        freq = xvals[xp.argmax(yvals)]
        valnew = xp.max(yvals)
        if valnew < val * (1 - 1e-7):
            pltfig1(xvals, yvals, addvline=(freq,), title=f"Optimize_1dfreq {i=} {val=} {valnew=}").show()
        assert valnew >= val * (1 - 1e-7), f"{val=} {valnew=} {i=} {val-valnew=}"
        if abs(valnew - val) < 1e-7: margin /= 4
        val = valnew
    return freq, val / xp.sum(xp.abs(sig2))

def pltfig(datas, title = None, yaxisrange = None, modes = None, marker = None, addvline = None, addhline = None, line_dash = None, fig = None, line=None):
    """
    Plot a figure with the given data and parameters.

    Parameters:
    datas : list of tuples
        Each tuple contains two lists or array-like elements, the data for the x and y axes.
        If only y data is provided, x data will be generated using xp.arange.
    title : str, optional
        The title of the plot (default is None).
    yaxisrange : tuple, optional
        The range for the y-axis as a tuple (min, max) (default is None).
    mode : str, optional
        The mode of the plot (e.g., 'line', 'scatter') (default is None).
    marker : str, optional
        The marker style for the plot (default is None).
    addvline : float, optional
        The x-coordinate for a vertical line (default is None).
    addhline : float, optional
        The y-coordinate for a horizontal line (default is None).
    line_dash : str, optional
        The dash style for the line (default is None).
    fig : matplotlib.figure.Figure, optional
        The figure object to plot on (default is None).
    line : matplotlib.lines.Line2D, optional
        The line object for the plot (default is None).

    Returns:
    None
    """
    if fig is None: fig = go.Figure(layout_title_text=title)
    elif title is not None: fig.update_layout(title_text=title)
    if not all(len(data) == 2 for data in datas): datas = [(xp.arange(len(data)), data) for data in datas]
    # if not(all(len(data) == 2 for data in datas) and all(type(to_device(data[0])) == xp.array and type(to_device(data[1])) == xp.array  for data in datas)):
    #     print(len(datas[0]))
    #     print(len(datas[1]))
    #     print(type(datas[0][0]))
    #     print(type(datas[0][1]))
    #     print(type(datas[1][0]))
    #     print(type(datas[1][1]))
    #     print(datas[0][0].dtype)
    #     print(datas[0][1].dtype)
    #     print(datas[1][0].dtype)
    #     print(datas[1][1].dtype)
    assert all(len(data) == 2 for data in datas) 
    assert all(isinstance(to_device(data[0]), xp.ndarray) and isinstance(to_device(data[1]), xp.ndarray) for data in datas)
    if modes is None:
        modes = ['lines' for _ in datas]
    elif isinstance(modes, str):
        modes = [modes for _ in datas]
    for idx, ((xdata, ydata), mode) in enumerate(zip(datas, modes)):
        if line == None and idx == 1: line = dict(dash='dash')
        fig.add_trace(go.Scatter(x=to_host(xdata), y=to_host(ydata), mode=mode, marker=marker, line=line))
        assert len(to_host(xdata)) == len(to_host(ydata))
    pltfig_hind(addhline, addvline, line_dash, fig, yaxisrange)
    return fig



def pltfig1(xdata, ydata, title = None, yaxisrange = None, mode = None, marker = None, addvline = None, addhline = None, line_dash = None, fig = None, line=None):
    """
    Plot a figure with the given data and parameters.

    Parameters:
    xdata : list or array-like or None
        The data for the x-axis.
        If is None, and only y data is provided, x data will be generated using xp.arange.
    ydata : list or array-like
        The data for the y-axis.
    title : str, optional
        The title of the plot (default is None).
    yaxisrange : tuple, optional
        The range for the y-axis as a tuple (min, max) (default is None).
    mode : str, optional
        The mode of the plot (e.g., 'line', 'scatter') (default is None).
    marker : str, optional
        The marker style for the plot (default is None).
    addvline : float, optional
        The x-coordinate for a vertical line (default is None).
    addhline : float, optional
        The y-coordinate for a horizontal line (default is None).
    line_dash : str, optional
        The dash style for the line (default is None).
    fig : matplotlib.figure.Figure, optional
        The figure object to plot on (default is None).
    line : matplotlib.lines.Line2D, optional
        The line object for the plot (default is None).

    Returns:
    None
    """
    if xdata is None: xdata = xp.arange(len(ydata))
    if fig is None: fig = go.Figure(layout_title_text=title)
    elif title is not None: fig.update_layout(title_text=title)
    if mode is None: mode = 'lines'
    fig.add_trace(go.Scatter(x=to_host(xdata), y=to_host(ydata), mode=mode, marker=marker, line=line))
    assert len(to_host(xdata)) == len(to_host(ydata))
    pltfig_hind(addhline, addvline, line_dash, fig, yaxisrange)
    return fig

def pltfig_hind(addhline, addvline, line_dash_in, fig, yaxisrange):
    if yaxisrange: fig.update_layout(yaxis=dict(range=yaxisrange), )
    if addvline is not None:
        if line_dash_in is None:
            line_dash = ['dash' for _ in range(len(addvline))]
            line_dash[0] = 'dot'
        elif isinstance(line_dash_in, str):
            line_dash = [line_dash_in for _ in range(len(addvline))]
        else: line_dash = line_dash_in
        for x, ldash in zip(addvline, line_dash): fig.add_vline(x=to_scalar(x), line_dash=ldash)
    if addhline is not None:
        if line_dash_in is None:
            line_dash = ['dash' for _ in range(len(addhline))]
            line_dash[0] = 'dot'
        elif isinstance(line_dash_in, str):
            line_dash = [line_dash_in for _ in range(len(addhline))]
        else: line_dash = line_dash_in
        for y, ldash in zip(addhline, line_dash): fig.add_hline(y=to_scalar(y), line_dash=ldash)


### Config.py

In [ ]:
# parser = argparse.ArgumentParser()
# parser.add_argument('--sf', type=int, default=10, help="Set the value of sf")
# args = parser.parse_args(args=[])

class Config:
    sf = 12
    bw = 406250
    sig_freq = 2.4e9
    preamble_len = 240
    payload_len = 70
    guess_f = -40000
    fs = 1e6
    skip_preambles = 8
    code_len = 2

    sfdpos = preamble_len + code_len
    sfdend = sfdpos + 3
    total_len = sfdend + payload_len

    cfo_range = bw // 4
    n_classes = 2 ** sf
    tsig = 2 ** sf / bw * fs  # in samples
    nsamp = around(n_classes * fs / bw)
    nsampf = (n_classes * fs / bw)

    tstandard = xp.linspace(0, nsamp / fs, nsamp + 1)[:-1]
    decode_matrix_a = xp.zeros((n_classes, nsamp), dtype=xp.complex64)
    decode_matrix_b = xp.zeros((n_classes, nsamp), dtype=xp.complex64)

    betai = bw / ((2 ** sf) / bw)
    # wflag = True
    # for code in range(n_classes):
    #     if (code-1)%4!=0 and sf>=11 and wflag:
    #         wflag = False
    #         continue
    #     nsamples = around(nsamp / n_classes * (n_classes - code))
    #     f01 = bw * (-0.5 + code / n_classes)
    #     refchirpc1 = xp.exp(-1j * 2 * xp.pi * (f01 * tstandard + 0.5 * betai * tstandard * tstandard))
    #     f02 = bw * (-1.5 + code / n_classes)
    #     refchirpc2 = xp.exp(-1j * 2 * xp.pi * (f02 * tstandard + 0.5 * betai * tstandard * tstandard))
    #     decode_matrix_a[code, :nsamples] = refchirpc1[:nsamples]
    #     if code > 0: decode_matrix_b[code, nsamples:] = refchirpc2[nsamples:]

    detect_range_pkts = 1000 # !!! TODO
    fft_n = int(fs)
    if USE_GPU:
        plan = xfft.get_fft_plan(xp.zeros(fft_n, dtype=xp.complex64))
        plan2 = xfft.get_fft_plan(xp.zeros(nsamp, dtype=xp.complex64))
    else:
        plan = None
        plan2 = None

### Reader.py

In [ ]:

class SlidingComplex64Reader: # TODO reader slow unbuffered
    dtype = xp.complex64
    itemsize = 8  # complex64

    def __init__(self, file_path):
        """
        Initializes the reader for a complex64 binary file.

        Args:
            file_path (str): The path to the binary file.
        """
        self.file_path = file_path

    def get(self, start, length):
        """
        Reads a chunk of data from the file.

        Args:
            start (int): The starting index (in terms of complex64 elements).
            length (int): The number of complex64 elements to read.

        Returns:
            xp.ndarray: An array containing the requested data.
        """
        byte_start = start * self.itemsize
        byte_length = length * self.itemsize

        try:
            with open(self.file_path, 'rb') as f:
                f.seek(byte_start)
                data_bytes = f.read(byte_length)

            if len(data_bytes) != byte_length:
                raise IOError(f"Read fewer bytes than expected. Expected {byte_length}, got {len(data_bytes)}")

            # Convert the bytes to a NumPy/CuPy array
            # Note: The .frombuffer method is efficient as it creates a view of the bytes.
            # We then copy it to the appropriate device (CPU or GPU) if necessary.
            data_array = xp.frombuffer(data_bytes, dtype=self.dtype)
            return data_array

        except FileNotFoundError:
            print(f"Error: The file '{self.file_path}' was not found.")
            return None
        except Exception as e:
            print(f"An error occurred: {e}")
            return None

### Main: Start Running

In [ ]:
# fstart = -41023.388364708379
# tstart =  4240090.873306715
file_path = "data/test_1226"

file_size = os.path.getsize(file_path)
complex64_size = xp.dtype(xp.complex64).itemsize
assert complex64_size == 8
print(f"{file_path=} Size in Number of symbols: {file_size // complex64_size // Config.nsamp}")

reader = SlidingComplex64Reader(file_path)

### Fitcoef: 

- Polynomial fit unwrapped phase of each symbol, generating 240 quadratic coefs 
    - Does not smooth the coef results
    - Fix beta, only change coeflist[:, 1] and coeflist[:, 2] via residue frequency and phase
- Compute phase difference between neighboring symbols 
    - Difference between two coefs at the tjump (time estimations from input)
    - Wrap into 2pi
    - Evaluate time difference that cause this (Use BW from input guessf, dt = dphase / estBW, estBW = Bw * (1 + estF / sigF))
    - Linear fit the time differences
    - Return new observations of tjump as a corrected coeft (len=2)


In [ ]:
def fitcoef2(coeff: xp.array, coeft: xp.array, reader: SlidingComplex64Reader):
    betai = Config.bw / ((2 ** Config.sf) / Config.bw) * xp.pi # frequency slope to phase 2d slope, *pi
    coeflist = []

    for pidx in range(0, Config.preamble_len):
        
        # compute coef2d_est2: polynomial curve fitting unwrapped phase of symbol pidx
        # time: tstart to tend
        # frequency at tstart: - estbw * 0.5 + estf
        estf = xp.polyval(coeff, pidx)
        estbw = Config.bw * (1 + estf / Config.sig_freq)
        beta1 = betai * (1 + 2 * estf / Config.sig_freq)
        tstart = xp.polyval(coeft, pidx)
        tend = xp.polyval(coeft, pidx + 1)
        beta2 = 2 * xp.pi * (- estbw * 0.5 + estf) - tstart * 2 * beta1
        coef2d_est2 = xp.array([to_scalar(beta1), to_scalar(beta2), 0])

        # align 3rd parameter of coef2d_est2 to observed phase at tstart
        nsymbr_start = math.ceil(tstart * Config.fs + Config.nsamp / 8)
        nsymbr_end = math.ceil(tend * Config.fs - Config.nsamp / 8)
        nsymbr = xp.arange(math.ceil(tstart * Config.fs + Config.nsamp / 8), math.ceil(tend * Config.fs - Config.nsamp / 8))
        tsymbr = nsymbr / Config.fs

        sig0 = reader.get(nsymbr_start, nsymbr_end - nsymbr_start)
        sig1 = sig0 * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))
        data0 = myfft(sig1, n=Config.fft_n, plan=Config.plan)
        freq1 = xp.fft.fftshift(xp.fft.fftfreq(Config.fft_n, d=1 / Config.fs))[xp.argmax(xp.abs(data0))]
        freq, valnew = optimize_1dfreq_fast(sig1, tsymbr, freq1, Config.fs / Config.fft_n * 5)
        # freqf, valnew = optimize_1dfreq(sig1, tsymbr, freq1, Config.fs / Config.fft_n * 5)
        # print(f"Initial freq offset: {freq1}, after FFT fit: {freq}, after precise fit: {freqf}, diff: {freqf - freq}")

        # adjust coef2d_est2[1] according to freq difference
        coef2d_est2[1] = 2 * xp.pi * (- estbw * 0.5 + estf + freq) - tstart * 2 * beta1
        sig2 = sig0 * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))
        # freq, valnew = optimize_1dfreq(sig2, tsymbr, freq1, Config.fs / Config.fft_n * 5)
        # print(f"{freq=} should be zero {valnew=}")
        coef2d_est2[2] += xp.angle(sig0.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))))
        # print(f"{xp.angle(sig0.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))))} should be zero")
        coeflist.append(coef2d_est2)

        if False:#pidx < 2:
            pltfig1(tsymbr, wrap(xp.angle(sig0) - xp.polyval(coef2d_est2, tsymbr)),
                    title=f"Fitted Phase Curve for Preamble Symbol {pidx}",
                    mode='lines').show()
            pltfig(((tsymbr, xp.unwrap(xp.angle(sig0))), (tsymbr, xp.polyval(coef2d_est2, tsymbr) - xp.polyval(coef2d_est2, tsymbr[0]) + xp.angle(sig0[0]))),
                    title=f"Fitted Phase Curve Comparison for Preamble Symbol {pidx}",
                    modes='lines').show()
    return xp.array(coeflist)



In [ ]:
def fitcoef4(coeff: xp.array, coeft: xp.array, reader: SlidingComplex64Reader):
    coeflist = fitcoef2(coeff, coeft, reader)
    
    # plot phase difference between consecutive symbols
    phasedifflist = xp.zeros((Config.preamble_len - 1,), dtype=xp.float32)
    for pidx in range(Config.preamble_len - 1):
        tjump = xp.polyval(coeft, pidx + 1)
        phasediff = wrap(xp.polyval(coeflist[pidx + 1], tjump) - xp.polyval(coeflist[pidx], tjump))
        phasedifflist[pidx] = phasediff
    phasedifflist_unwrap = xp.unwrap(xp.array(phasedifflist))
    # pltfig1(range(Config.preamble_len - 1), phasedifflist_unwrap, title="plot phase difference between consecutive symbols").show()

    # fit a line to phase difference to estimate cfo and time drift
    tdifflist = xp.zeros_like(phasedifflist_unwrap)
    for pidx in range(Config.preamble_len - 1):
        estbw = Config.bw * (1 + xp.polyval(coeff, pidx + 0.5) / Config.sig_freq)
        tdifflist[pidx] = phasedifflist_unwrap[pidx] / 2 / xp.pi / estbw # phasediff is caused by mismatched symbol change time, -> bw mismatch. phase = 2pi * bw * dt
    xrange = xp.arange(50, len(tdifflist)) # !!! ignore first 50 points
    tdiff_coef = xp.polyfit(xrange, tdifflist[xrange], 1)
    coeft_new = coeft.copy()
    coeft_new[-2:] += tdiff_coef
    print(f"{tdiff_coef=} {coeft=} {coeft_new=} cfo ppm from time: {1 - coeft_new[0] / Config.nsampf * Config.fs} cfo: {(1 - coeft_new[0] / Config.nsampf * Config.fs) * Config.sig_freq} unwrapped phasediff so t may shift by margin {1/Config.bw}")

    return coeft_new

In [ ]:
# tsymblen = 2 ** Config.sf / Config.bw * (1 - fstart / Config.sig_freq)

coeff = xp.array((-0.512392321665, -41023.388364708379))
coeft = xp.array((1.01716420e-13, 1.00826333e-02, 4.24009137e+00))

# coeff = xp.array((0, fstart))
# coeft = xp.array((tsymblen, to_scalar(tstart) / Config.fs))
coeft = fitcoef4(coeff, coeft, reader)
# coeft = fitcoef4(coeff, coeft, reader) # do this to check correctness of fitcoef4

### get coeflist from renewed coeft, and Find intersections between each coef

### findintersections.py

In [ ]:

def find_intersections(coefa, coefb, tstart2, reader, epsilon, pidx: int, remove_range):
    """
    Finds all intersection points of two quadratic polynomials within a specified range.

    Args:
        coefa (xp.ndarray): Coefficients of the first quadratic polynomial [c2, c1, c0].
        coefb (xp.ndarray): Coefficients of the second quadratic polynomial [c2, c1, c0].
        tstart2 (float): The center of the search range.
        epsilon (float): The half-width of the search range.

    Returns:
        xp.ndarray: A sorted array of the x-coordinates of the intersection points.
    """
    x_min = tstart2 - epsilon
    x_max = tstart2 + epsilon

    # Compute the difference polynomial coefa - coefb
    poly_diff = xp.polysub(coefa, coefb)

    # The difference polynomial is also a quadratic: poly_diff(x) = ax^2 + bx + c
    a, b, c = poly_diff

    # Find the vertex of the difference polynomial, if it exists within the range
    if a != 0:
        x_vertex = -b / (2 * a)
        y_vertex = xp.polyval(poly_diff, x_vertex)
    else:  # The difference is a linear function
        x_vertex = None
        y_vertex = None
    y_min_bound = xp.polyval(poly_diff, x_min)
    y_max_bound = xp.polyval(poly_diff, x_max)

    # Determine the range of y-values for the difference polynomial within [x_min, x_max]
    y_values = [y_min_bound, y_max_bound]
    if x_min <= x_vertex <= x_max and a != 0:
        y_values.append(y_vertex)

    y_lower = min(y_values)
    y_upper = max(y_values)

    n_min = ceil((y_lower - xp.pi) / (2 * xp.pi))
    n_max = floor((y_upper - xp.pi) / (2 * xp.pi))

    nrange = xp.arange(int(n_min), int(n_max) + 1) * 2 * xp.pi

    roots = []
    if a == 0 and b == 0:  
        return xp.array([])
    elif a == 0:  
        for n in nrange:
            root = (n - c) / b
            if x_min <= root <= x_max:
                roots.append(root)
    else:
        roots = []
        for n in nrange:
            shifted_c = c - n
            discriminant = b**2 - 4 * a * shifted_c
            if discriminant >= 0:
                sqrt_discriminant = xp.sqrt(discriminant)
                root1 = (-b + sqrt_discriminant) / (2 * a)
                root2 = (-b - sqrt_discriminant) / (2 * a)
                if x_min <= root1 <= x_max:
                    roots.append(root1)
                if x_min <= root2 <= x_max:
                    roots.append(root2)
    intersection_points = xp.array(roots)

    xv = to_device(xp.arange(ceil(x_min * Config.fs), ceil(x_max * Config.fs), dtype=int))
    sig = reader.get(to_scalar(xv[0]), len(xv))
    sig_ref1 = sig * xp.exp(-1j * xp.polyval(coefa, xv / Config.fs))
    sig_ref2 = sig * xp.exp(-1j * xp.polyval(coefb, xv / Config.fs))
    
    if len(intersection_points) != 0:
        selected = max(intersection_points, key=lambda x: xp.abs(xp.sum(sig_ref1[:ceil(x * Config.fs - xv[0])])) + xp.abs(xp.sum(sig_ref2[ceil(x * Config.fs - xv[0]):])))
        selected2 = min(intersection_points, key=lambda x: abs(x - tstart2))
    else: return None

    if False:#pidx < 5:
        x_vals = xp.linspace(x_min, x_max, 400)
        y_vals_a = wrap(xp.polyval(coefa, x_vals))
        y_vals_b = wrap(xp.polyval(coefb, x_vals))
        fig = pltfig1(x_vals, y_vals_a)
        pltfig1(x_vals, y_vals_b, fig=fig)
        pltfig1(xv / Config.fs, xp.angle(sig), fig=fig, mode='markers', marker=dict(color='green', symbol='x', size=8))
        fig.add_trace(go.Scatter(x=to_host(intersection_points), y=to_host(wrap(xp.polyval(coefa, intersection_points))), mode='markers', marker=dict(color='red', symbol='circle', size=10)))
        fig.add_trace(go.Scatter(x=to_host(intersection_points), y=to_host(wrap(xp.polyval(coefb, intersection_points))), mode='markers', marker=dict(color='red', symbol='circle', size=10)))
        fig.add_trace(go.Scatter(x=[to_scalar(selected)], y=[to_scalar(wrap(xp.polyval(coefa, selected)))], mode='markers', marker=dict(color='blue', symbol='cross', size=10)))
        fig.add_hline(y=to_scalar(x_min), line_dash='dash')
        fig.add_hline(y=to_scalar(x_max), line_dash='dash')
        fig.add_hline(y=to_scalar(tstart2), line_dash='dash')
        fig.update_layout(title_text=f'Intersection Points of Two Quadratic Polynomials {pidx=}')
        fig.show()
        
    if selected2 != selected:
        print(f"find_intersections(): break point not closeset to tstart2 selected {selected - tstart2 =}")
        if remove_range: return None
    return selected


### find intersections

In [ ]:
coeflist = fitcoef2(coeff, coeft, reader)

In [ ]:
# pltfig1(None, coeflist[:, 0] - xp.polyval(xp.polyfit(xp.arange(Config.preamble_len), coeflist[:, 0], 1), xp.arange(Config.preamble_len)), title="Coefficient a over Preamble Symbols (detrended)", mode='lines').show()
# pltfig1(None, coeflist[:, 1] - xp.polyval(xp.polyfit(xp.arange(Config.preamble_len), coeflist[:, 1], 1), xp.arange(Config.preamble_len)), title="Coefficient b over Preamble Symbols (detrended)", mode='lines').show()
# pltfig1(None, coeflist[:, 2], title="Coefficient c over Preamble Symbols", mode='lines').show()
        

In [ ]:

sec_xlist = []
sec_tlist = []
if True:
    for pidx in xp.arange(Config.preamble_len):
        tstart2 = xp.polyval(coeft, pidx)
        if pidx > 0:
            selected = find_intersections(coeflist[pidx - 1], coeflist[pidx], tstart2, reader, 1e-5, pidx, remove_range=False) #!!! TODO remove range
        else:
            selected = find_intersections(xp.zeros_like(coeflist[pidx]), coeflist[pidx], tstart2, reader, 1e-5, pidx, remove_range=False) #!!! TODO remove range
        if selected != None:
            sec_xlist.append(pidx)
            sec_tlist.append(to_scalar(selected))
    sec_xlist = xp.array(sec_xlist)
    sec_tlist = xp.array(sec_tlist)
    with open(f"intersections0.pkl","wb") as f:
        pickle.dump((sec_xlist, sec_tlist), f)

with open(f"intersections0.pkl","rb") as f:
    sec_xlist, sec_tlist = pickle.load(f)
coeff_time = xp.polyfit(sec_xlist, sec_tlist, 1)
# pltfig(((sec_xlist, sec_tlist), (sec_xlist, xp.polyval(coeff_time, sec_xlist))),
#        title="Intersection Points and Fitted Line",
#        modes=('markers', 'lines'),
#        line_dash='dash').show()
# pltfig1(sec_xlist, sec_tlist - xp.polyval(coeff_time, sec_xlist),
#         title="Residuals of Intersection Points from Fitted Line",
#         mode='markers').show()

### Plot the intersection point results, fit tjump

In [ ]:
coeflist = fitcoef2(coeff, coeft, reader)
# sec_xlist2 = []
# sec_tlist2 = []
# for pidx in xp.arange(6):
#     x1 = ceil(xp.polyval(coeff_time, pidx + 0.8) * Config.fs)
#     x2 = ceil(xp.polyval(coeff_time, pidx + 1) * Config.fs)
#     nsymbr = xp.arange(x1, x2)
#     tsymbr = nsymbr / Config.fs
#     sig = reader.get(x1, x2 - x1)
#     res = sig.dot(xp.exp(-1j * xp.polyval(coeflist[pidx], tsymbr)))
# if True:
#     for pidx in xp.arange(4, 5):
#         tstart2 = xp.polyval(coeft, pidx)
#         selected = find_intersections(coeflist[pidx - 1], coeflist[pidx], tstart2, reader, 1e-4, pidx, remove_range=False)
#         if selected != None:
#             sec_xlist2.append(pidx)
#             sec_tlist2.append(to_scalar(selected))
#     sec_xlist2 = xp.array(sec_xlist2)
#     sec_tlist2 = xp.array(sec_tlist2)
#     # with open(f"intersections0.pkl","wb") as f:
#         # pickle.dump((sec_xlist, sec_tlist), f)

# # with open(f"intersections0.pkl","rb") as f:
#     # sec_xlist, sec_tlist = pickle.load(f)
# coeff_time2 = xp.polyfit(sec_xlist2, sec_tlist2, 1)
# print(f"guessed: {coeft=} coeff_time={coeff_time2[0]:.12f},{coeff_time2[1]:.12f} cfo ppm from time: {1 - coeff_time2[0] / Config.nsampf * Config.fs} cfo: {(1 - coeff_time2[0] / Config.nsampf * Config.fs) * Config.sig_freq}")
# pltfig(((sec_xlist2, sec_tlist2), (sec_xlist2, xp.polyval(coeff_time2, sec_xlist2))), title="intersect points fitline").show()
# pltfig1(sec_xlist2, sec_tlist2 - xp.polyval(coeff_time2, sec_xlist2), title="intersect points diff").show()


In [ ]:
print(f"guessed: {coeft=} coeff_time={coeff_time[0]:.12f},{coeff_time[1]:.12f} cfo ppm from time: {1 - coeff_time[0] / Config.nsampf * Config.fs} cfo: {(1 - coeff_time[0] / Config.nsampf * Config.fs) * Config.sig_freq}")
pltfig(((sec_xlist, sec_tlist), (sec_xlist, xp.polyval(coeff_time, sec_xlist))), title="intersect points fitline").show()
pltfig1(sec_xlist, sec_tlist - xp.polyval(coeff_time, sec_xlist), title="intersect points diff").show()

sec_tdiff_list = sec_tlist - xp.polyval(coeff_time, sec_xlist)
sec_smoothed_tlist = sec_tlist.copy()
for pidx in range(1, len(sec_tdiff_list) - 1):
    if abs(sec_tdiff_list[pidx] - sec_tdiff_list[pidx-1]) > 0.2e-6 and abs(sec_tdiff_list[pidx] + sec_tdiff_list[pidx-1]) > 0.2e-6:
        sec_tdiff_list[pidx] = (sec_tdiff_list[pidx-1] + sec_tdiff_list[pidx+1])/2
        sec_smoothed_tlist[pidx] = (sec_smoothed_tlist[pidx-1] + sec_smoothed_tlist[pidx+1])/2

coeff_time_error = xp.polyfit(sec_xlist, sec_tdiff_list, 1)
pltfig(((sec_xlist, sec_tdiff_list), (sec_xlist, xp.polyval(coeff_time_error, sec_xlist))),
       title="intersect points fit on difference").show()
pltfig1(sec_xlist, sec_tdiff_list - xp.polyval(coeff_time_error, sec_xlist), title="intersect points diff 2").show()
print(f"coeff_time_error={coeff_time_error[0]:.12f},{coeff_time_error[1]:.12f}")


- The cell below may be errorsome.
- There is no visible curve on the previous image, so coeff_time3 if using polyfit(_,_,2) the large error of time itself may not be able to generate a good x^2 parameter.

```python
coeff_time31 = xp.polyfit(sec_xlist, sec_smoothed_tlist, 1)
pltfig(((sec_xlist, sec_smoothed_tlist), (sec_xlist, xp.polyval(coeff_time31, sec_xlist))),
       title="intersect points fitline by linear coeff_time31").show()
pltfig1(sec_xlist, sec_smoothed_tlist - xp.polyval(coeff_time31, sec_xlist), title="intersect points linear diff coeff_time31").show()

coeff_time3 = xp.polyfit(sec_xlist, sec_smoothed_tlist, 2)
pltfig(((sec_xlist, sec_smoothed_tlist), (sec_xlist, xp.polyval(coeff_time3, sec_xlist))),
       title="intersect points fitline by quad coeff_time3").show()
pltfig1(sec_xlist, sec_smoothed_tlist - xp.polyval(coeff_time3, sec_xlist), title="intersect points quad diff coeff_time3").show()
print(f"{coeff_time3=}")
# t1 - coef(1) - coef(0) = a + b
freq_start = (1 - (coeff_time3[0] + coeff_time3[1]) / (2 ** Config.sf / Config.bw)) * Config.sig_freq
# t2 - t1 = coef(2) - coef(1) - coef(1) + coef(0) = 2a
freq_rate = - 2 * coeff_time3[0] / (2 ** Config.sf / Config.bw) * Config.sig_freq
print(f"{freq_start=} {freq_rate=}")
```

```python
pidx_range = xp.arange(Config.preamble_len)
pltfig1(pidx_range, xp.polyval(coeft, pidx_range) - xp.polyval(coeff_time3, pidx_range), title="time difference of new and old estimation").show()
```

### Estimate Freq

In [ ]:

coeff_time31 = xp.polyfit(sec_xlist, sec_smoothed_tlist, 1)
print(f"{coeff_time31=}")

coeff_time3 = xp.polyfit(sec_xlist, sec_smoothed_tlist, 2)
freq_start = (1 - (coeff_time3[0] + coeff_time3[1]) / (2 ** Config.sf / Config.bw)) * Config.sig_freq
freq_rate = - 2 * coeff_time3[0] / (2 ** Config.sf / Config.bw) * Config.sig_freq # frequency change rate, estimated from time change rate

beta = Config.bw / ((2 ** Config.sf) / Config.bw) * (1 + 2 * freq_start / Config.sig_freq)
print("beta=", beta)
sigt = 2 ** Config.sf / Config.bw * (1 + freq_start / Config.sig_freq)
print(f"symbol duration: {sigt} s")
print(f"time diff caused by {coeff_time3[0]=} over symbol duration: { xp.polyval(coeff_time3, Config.preamble_len) - xp.polyval(coeff_time3[1:], Config.preamble_len)} s")
print(f"freq diff caused by time drift over symbol duration: { - beta * (xp.polyval(coeff_time3, Config.preamble_len) - xp.polyval(coeff_time3[1:], Config.preamble_len))} Hz, neglegible")
pltfig(((sec_xlist, xp.polyval(coeff_time3, sec_xlist) - sec_smoothed_tlist), (sec_xlist, xp.polyval(coeff_time31, sec_xlist) - sec_smoothed_tlist)), title=f"intersection fit error 2d/1d").show()
coeff_time3 = xp.hstack([0, coeff_time31[0], coeff_time31[1]]) # ~!!! TODO

# TODO TODO1 Freq and Jumptime not corresponding

- output estcoef: the instantaneous frequencies at junction of symbols, evaluated from coeflist

In [ ]:
pidx_range = xp.arange(Config.preamble_len)
# TODO simplify
estcoefs = []
for ixx in range(2):
    print(f"start computing {'start' if ixx == 0 else 'end'} frequencies from coeflist and tjump")
    dd = []
    for pidx in range(240):
        estf = xp.polyval(coeff, pidx)
        if ixx == 0:
            bwdiff = -Config.bw * (1 + estf / Config.sig_freq) / 2
        else:
            bwdiff = Config.bw * (1 + estf / Config.sig_freq) / 2
        dd.append(to_scalar((coeflist[pidx, 0] * 2 * xp.polyval(coeff_time3, pidx + ixx) + coeflist[pidx, 1]) / 2 / xp.pi - bwdiff))
    dd = xp.array(dd)
    pidx_range2 = xp.arange(50, Config.preamble_len - 10)
    estcoef = xp.polyfit(pidx_range2, dd[pidx_range2], 1)
    intercept = xp.mean(dd[pidx_range2] - freq_rate * pidx_range2)
    estcoefs.append(estcoef)

    pltfig(((pidx_range2, dd[pidx_range2]), (pidx_range2, xp.polyval(estcoef, pidx_range2))),
           title=f"intersect points fitline freq{ixx} {estcoef=}").show()
    pltfig1(pidx_range2, dd[pidx_range2] - xp.polyval(estcoef, pidx_range2), title=f"intersect points diff freq{ixx}").show()
    
    fdiff = intercept - estcoef[1] # freq = (2at + b) / 2pi deltaf = a/pi deltat
    tdiff =  fdiff / xp.mean(coeflist[:, 0]) * xp.pi
    print(f"new computation: estcoef at t=0: {estcoef[1]:.12f} estf change rate per symb: {estcoef[0]:.12f} old estimation from tdiff: {freq_rate:.12f} {intercept:.12f} {tdiff:.12f}")

    # f(x) = a x + b
    # f(x + 0.5) = a x + 0.5a + b
    # t(x) = T ( x - f(0)/F - f(1)/F - ... - f(n)/F)
    # t(x) = T ( x - x(f(0)+f(x)/2)/F)
    # t(x) = T ( x - x(ax+b+b)/2/F)
    # t(x) = T (ax^2/2F + x (b/F + 1))
    tsig = 2 ** Config.sf / Config.bw
    assert len(estcoef) == 2
    estcoeft = xp.hstack([estcoef[0] / 2 / Config.sig_freq * tsig, ((estcoef[0] + estcoef[1]) / Config.sig_freq + 1) * tsig, coeft[-1]])
    print(f"t from f {estcoeft=}, {coeff_time3=}")
    # pltfig1(pidx_range2, xp.polyval(estcoeft, pidx_range2) - xp.polyval(coeff_time3, pidx_range2), title="time difference of new and old estimation").show()

    # compute tdiff
    # t(x) = ax^2 + bx + c
    # t(x+1)-t(x) = a(2x+1) + b = 2ax + (a + b)
    assert len(coeff_time3) == 3
    coeff_tlen = xp.hstack((coeff_time3[0] * 2, coeff_time3[0] + coeff_time3[1]))
    # f from t: t = T * (1 - f / F)
    # f = F * (1 - t / T) 
    # f = F * (1 - (a x + b) / T) = - a x F / T + F * (1 - b / T)
#    coeff_from_tlen = xp.hstack((-  Config.sig_freq / tsig * coeff_tlen[0], Config.sig_freq * (1 - coeff_tlen[1] / tsig)))
    # -2aF/T, F(1- (a+b)/T)
    coeff_from_tlen = xp.hstack([- coeff_time3[0] * 2 * Config.sig_freq / tsig, Config.sig_freq * (1 - xp.sum(coeff_time3[:2]) / tsig)])

    print(f"{coeff_tlen=} {coeff_from_tlen=} {estcoef=}") 
    
    

```python
coeff_from_tlen = xp.hstack((-  Config.sig_freq / tsig * coeff_tlen[0], Config.sig_freq * (1 - coeff_tlen[1] / tsig)))
```
if replace to 
```python
coeff_from_tlen = xp.hstack((-  2*Config.sig_freq / tsig * coeff_tlen[0], Config.sig_freq * (1 - coeff_tlen[1] / tsig)))
```
its similar to estcoef

TODOhere

In [ ]:
pidx_range = xp.arange(Config.preamble_len)
pltfig(((pidx_range, xp.polyval(coeff_from_tlen, pidx_range)), (pidx_range, xp.polyval(estcoef, pidx_range))), title="comparison of coeff hypothesized from time offset and measured cfos from jump point frequencies").show()
pltfig1(pidx_range, xp.polyval(coeff_from_tlen, pidx_range) - xp.polyval(estcoef, pidx_range), title="comparison of coeff hypothesized from time offset and measured cfos from jump point frequencies").show()


In [ ]:
# Testing 

for repeat in range(2):
    estf_error_vals = xp.zeros((Config.preamble_len,), dtype=xp.float32)
    estf_power = xp.zeros((Config.preamble_len,), dtype=xp.float32)
    estf_corrected_power = xp.zeros((Config.preamble_len,), dtype=xp.float32)
    for pidx in range(0, Config.preamble_len):
        # if pidx % 20 != 0: continue
        if repeat == 0:
            freq_start = xp.polyval(coeff_from_tlen, pidx)
            freq_end = xp.polyval(coeff_from_tlen, pidx + 1)
        else:
            freq_start = xp.polyval(estcoef, pidx)
            freq_end = xp.polyval(estcoef, pidx + 1)
        # print(f"{freq_start=} {freq_old=} {Config.bw/2=}")
        est_f1 = freq_start - Config.bw / 2 * (1 + freq_start / Config.sig_freq)
        est_f2 = freq_end + Config.bw / 2 * (1 + freq_end / Config.sig_freq)
        est_t1 = xp.polyval(coeff_time3, pidx) 
        est_t2 = xp.polyval(coeff_time3, pidx + 1) 
        est_beta = (est_f2 - est_f1) / (est_t2 - est_t1)
        est_coef2d = xp.hstack([est_beta * xp.pi, 2 * xp.pi * est_f1 - est_t1 * 2 * est_beta * xp.pi, 0])
        nsymbr_start = math.ceil(est_t1 * Config.fs + Config.nsamp / 8)
        nsymbr_end = math.ceil(est_t2 * Config.fs - Config.nsamp / 8)
        nsymbr = xp.arange(math.ceil(est_t1 * Config.fs + Config.nsamp / 8), math.ceil(est_t2 * Config.fs - Config.nsamp / 8))
        tsymbr = nsymbr / Config.fs

        sig0 = reader.get(nsymbr_start, nsymbr_end - nsymbr_start)
        # pltfig(((tsymbr, xp.unwrap(xp.angle(sig0))), (tsymbr, xp.polyval(est_coef2d, tsymbr) - xp.polyval(est_coef2d, tsymbr[0]) + xp.angle(sig0[0]))), title=f"Fit with predetermined values {pidx=} ").show()
        # pltfig1(tsymbr, xp.unwrap(xp.angle(sig0)) -  xp.polyval(est_coef2d, tsymbr) + xp.polyval(est_coef2d, tsymbr[0]) - xp.angle(sig0[0]), title=f"Fit with predetermined values diff {pidx=} ").show()
        sig1 = sig0 * xp.exp(-1j * xp.polyval(est_coef2d, tsymbr))
        data0 = myfft(sig1, n=Config.fft_n, plan=Config.plan)
        freq1 = xp.fft.fftshift(xp.fft.fftfreq(Config.fft_n, d=1 / Config.fs))[xp.argmax(xp.abs(data0))]
        freq, valnew = optimize_1dfreq_fast(sig1, tsymbr, freq1, Config.fs / Config.fft_n * 5) # 5hz margin
        # print(f"{freq1=} {freq=} {valnew=}")
        est_coef2d[1] = 2 * xp.pi * (est_f1 + freq) - est_t1 * 2 * est_beta * xp.pi
        est_coef2d[2] += xp.angle(sig0.dot(xp.exp(-1j * xp.polyval(est_coef2d, tsymbr))))
        # pltfig(((tsymbr, xp.unwrap(xp.angle(sig0))), (tsymbr, xp.polyval(est_coef2d, tsymbr) - xp.polyval(est_coef2d, tsymbr[0]) + xp.angle(sig0[0]))), title=f"Fit with predetermined values {pidx=} {freq=}").show()
        # pltfig1(tsymbr, xp.unwrap(xp.angle(sig0)) -  xp.polyval(est_coef2d, tsymbr) + xp.polyval(est_coef2d, tsymbr[0]) - xp.angle(sig0[0]), title=f"Fit with predetermined values diff {pidx=} {freq=}").show()
        estf_error_vals[pidx] = freq
        estf_power[pidx] = xp.abs(xp.sum(sig1))/xp.sum(xp.abs(sig0))
        estf_corrected_power[pidx] = valnew
    pltfig1(None, estf_error_vals, title=f"errors of frequency {'coeff_from_tlen' if repeat == 0 else 'estcoef'}").show()
    pltfig(((pidx_range, estf_power), (pidx_range, estf_corrected_power)), title=f"errors of frequency {'coeff_from_tlen' if repeat == 0 else 'estcoef'}").show()




        # betai = Config.bw / ((2 ** Config.sf) / Config.bw) * xp.pi
        
        # # compute coef2d_est2: polynomial curve fitting unwrapped phase of symbol pidx
        # # time: tstart to tend
        # # frequency at tstart: - estbw * 0.5 + estf
        # estf = freq_start
        # estbw = Config.bw * (1 + estf / Config.sig_freq) 
        # for betaC in xp.arange(0, 3, 0.5):
        #     beta1 = betai * (1 + betaC * estf / Config.sig_freq)
        #     tstart = xp.polyval(coeff_time3, pidx)
        #     tend = xp.polyval(coeff_time3, pidx + 1)
        #     beta2 = 2 * xp.pi * (- estbw * 0.5 + estf) - tstart * 2 * beta1
        #     coef2d_est2 = xp.array([to_scalar(beta1), to_scalar(beta2), 0])

        #     # align 3rd parameter of coef2d_est2 to observed phase at tstart
        #     nsymbr_start = math.ceil(tstart * Config.fs + Config.nsamp / 8)
        #     nsymbr_end = math.ceil(tend * Config.fs - Config.nsamp / 8)
        #     nsymbr = xp.arange(math.ceil(tstart * Config.fs + Config.nsamp / 8), math.ceil(tend * Config.fs - Config.nsamp / 8))
        #     tsymbr = nsymbr / Config.fs

        #     sig0 = reader.get(nsymbr_start, nsymbr_end - nsymbr_start)
        #     sig1 = sig0 * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))
        #     data0 = myfft(sig1, n=Config.fft_n, plan=Config.plan)
        #     freq1 = xp.fft.fftshift(xp.fft.fftfreq(Config.fft_n, d=1 / Config.fs))[xp.argmax(xp.abs(data0))]
        #     freq, valnew = optimize_1dfreq(sig1, tsymbr, freq1, Config.fs / Config.fft_n * 5)
        #     print(f"{freq1=} {freq=} {valnew=}")
        #     # freqf, valnew = optimize_1dfreq(sig1, tsymbr, freq1, Config.fs / Config.fft_n * 5)

        #     # adjust coef2d_est2[1] according to freq difference
        #     coef2d_est2[1] = 2 * xp.pi * (- estbw * 0.5 + estf + freq) - tstart * 2 * beta1
        #     sig2 = sig0 * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))
        #     freq, valnew = optimize_1dfreq(sig2, tsymbr, freq1, Config.fs / Config.fft_n * 5)
        #     print(f"{freq=} should be zero {valnew=} {betaC=}")
        #     coef2d_est2[2] += xp.angle(sig0.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))))
        #     print(coef2d_est2, coeflist[pidx])
        #     print(f"{(coef2d_est2[0] * 2 * tstart + coef2d_est2[1]) / 2 / xp.pi} {(coeflist[pidx][0] * 2 * tstart + coeflist[pidx][1]) / 2 / xp.pi}")
        #     # print(f"{xp.angle(sig0.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))))} should be zero")


In [ ]:
# pltfig1(None, coeflist[:, 0], title="coeflist[:, 0]").show()
# pltfig1(None, coeflist[:, 1], title="coeflist[:, 1]").show()
# pltfig1(None, coeflist[:, 2], title="coeflist[:, 2]").show()

### Connect phases
- input: coeff_time3, estcoef
- estimate all coefficients from coefs in time and freq, no original coeflist involved
    - however coeff_time3 and estcoef are not connected, estcoef is still fitted from original coeflist
- connect all phases together

In [ ]:
# TODO find a f and a t that minimizes power loss over all symbols, however, f contributes larger to power loss, t mainly affect phase. beta * deltaT(0.3us) = 12Hz, 100%powerloss=50Hz
# first disregard phase diff information, phase diff may be result of errorsome starting points.

# f changes linearly over time: estcoef[0], estcoef[1]
# tlen results from f + start time
# need to reduce size of F to reduce large-number computation error

In [ ]:


coeff_time = coeff_time3 # todo!!!2
# coeff_time[-1] += 0.4e-6 + 130e-9

# print(f"{xp.polyval(coeft, Config.preamble_len)=} {xp.polyval(coeff_time3, Config.preamble_len)=}")
# coeff_time = coeff_time3 # !!! todo !!!
# print(f"{coeff_time=} already replaced by coefftime3")

betai = Config.bw / ((2 ** Config.sf) / Config.bw) * xp.pi
coeffitlist = xp.zeros((Config.preamble_len, 3), dtype=xp.float64)
coeffitlist[:, 0] = betai * (1 + 2 * xp.polyval(estcoef, pidx_range) / Config.sig_freq)

bwdiff = - Config.bw * (1 + estcoef[1] / Config.sig_freq) / 2
coeffitlist[:, 1] = 2 * xp.pi * xp.polyval(estcoef, pidx_range) - xp.polyval(coeff_time, pidx_range) * 2 * coeffitlist[:, 0] + bwdiff * 2 * xp.pi

for pidx in pidx_range[1:]:
    coeffitlist[pidx, 2] -= wrap(xp.polyval(coeffitlist[pidx], xp.polyval(coeff_time, pidx)) - xp.polyval(coeffitlist[pidx - 1], xp.polyval(coeff_time, pidx)))


### Decode_core

- compute phases and power of 240 preambles

In [ ]:
codephase = xp.zeros((Config.total_len,), dtype=xp.float64)
powers = xp.zeros((Config.total_len,), dtype=xp.float64)
codephase_secondary = xp.zeros((Config.total_len,), dtype=xp.float64)

# preamble codephase and powers
for pidx in range(Config.preamble_len):
    x1 = ceil(xp.polyval(coeff_time, pidx) * Config.fs)
    x2 = ceil(xp.polyval(coeff_time, pidx + 1) * Config.fs)
    nsymbr = xp.arange(x1, x2)
    tsymbr = nsymbr / Config.fs
    sig = reader.get(x1, x2 - x1)
    res = sig.dot(xp.exp(-1j * xp.polyval(coeffitlist[pidx], tsymbr)))
    codephase[pidx] = xp.angle(res)
    powers[pidx] = xp.abs(res) / xp.sum(xp.abs(sig))
    if pidx in [0, 120, Config.preamble_len - 1]:
        # coeffitlist_comp = coeffitlist[pidx].copy()
        # coeffitlist_comp[2] += xp.angle(sig[0]) - xp.polyval(coeffitlist[pidx], tsymbr[0])
        print(f"{coeffitlist[pidx]=} {xp.polyval(coeffitlist[pidx], tsymbr[0])=} {xp.unwrap(xp.angle(sig))[-1]=} {xp.polyval(coeffitlist[pidx], tsymbr[-1]) - xp.polyval(coeffitlist[pidx], tsymbr[0]) + xp.angle(sig[0])=}")
        pltfig((
            (tsymbr, xp.unwrap(xp.angle(sig)) ), 
            (tsymbr, xp.polyval(coeffitlist[pidx], tsymbr) - xp.polyval(coeffitlist[pidx], tsymbr[0]) + xp.angle(sig[0]))), title=f"preamble codephase {pidx=} angle={xp.angle(res)} pow={xp.abs(res)/xp.sum(xp.abs(sig))}").show()
        pltfig1(tsymbr, xp.unwrap(xp.angle(sig)) - xp.polyval(coeffitlist[pidx], tsymbr) + xp.polyval(coeffitlist[pidx], tsymbr[0]) - xp.angle(sig[0]), title=f"preamble codephase {pidx=} fitdiff").show()
pltfig1(xp.arange(Config.preamble_len), xp.unwrap(codephase[:Config.preamble_len]), title="unwrap phase").show()
pltfig1(xp.arange(Config.preamble_len), powers[:Config.preamble_len], title="powers").show()

- see if missed a symbol

In [ ]:
fig=None

pidx = -1
x1 = math.ceil(xp.polyval(coeff_time, pidx) * Config.fs)
x2 = math.ceil(xp.polyval(coeff_time, pidx + 2) * Config.fs)
print(x1, x2)
nsymbr = xp.arange(x1, x2)
sig = reader.get(x1, x2 - x1)
pltfig1(None, xp.unwrap(xp.angle(sig)), title = "Pidx=-1 and Pidx 0").show()



### Expand Coeflist into two rows

In [ ]:
coeffitlist = xp.concatenate((coeffitlist, xp.zeros_like(coeffitlist)), axis=0) # 2, 240, 3, 2
coeffitlist = xp.stack((coeffitlist, xp.zeros_like(coeffitlist)), axis=0) # 2, 240, 3, 2
print(coeffitlist.shape)
coeffitlist[1, :, -1] = coeffitlist[0, :, -1] # share the last value as continuous phase

### Fit 241, 242 code

- input: coeff_time, estcoef
- fit with phase continuity using curves predicted, no optimizing
- the following block only computes code

In [ ]:

startphase = 0#xp.polyval(coeffitlist[Config.preamble_len + 4], xp.polyval(coeff_time, Config.preamble_len + 5 - 0.75))

codexdiffs = []
codes = xp.zeros((Config.total_len,), dtype=int)
for pidx in range(Config.sfdend, Config.total_len):
    tstart = xp.polyval(coeff_time, pidx - 0.75)
    tend = xp.polyval(coeff_time, pidx + 1 - 0.75)
    x1 = math.ceil(tstart * Config.fs)
    x2 = math.ceil(tend * Config.fs)
    nsymbr = xp.arange(x1, x2)
    sig = reader.get(x1, x2 - x1)
    if xp.mean(xp.abs(sig)) < 0.01:
        print(f"{pidx=} {xp.mean(xp.abs(sig))=} too small. is symbol ending? quitting, payload_len={pidx - Config.sfdend}")
        break
    
    x1 = math.ceil(tstart * Config.fs)
    x2 = math.ceil(tend * Config.fs)
    nsymbr = xp.arange(x1, x2)
    tsymbr = nsymbr / Config.fs

    # pltfig1(tsymbr, xp.unwrap(xp.angle(reader.get(x1, x2-x1))), title=f"{pidx=}").show()
    # pltfig1(tsymbr, xp.abs(reader.get(x1, x2-x1)), title=f"{pidx=}").show()
    assert xp.mean(xp.abs(reader.get(x1, x2-x1))) > 0.1, f"{pidx=} {xp.mean(xp.abs(reader.get(x1, x2-x1)))=} too small. is symbol ending?"
    estcoef_this = xp.polyval(coeff, pidx)

    beta1 = Config.bw / ((2 ** Config.sf) / Config.bw) * xp.pi * (1 + 2 * estcoef_this / Config.sig_freq)
    estbw = Config.bw * (1 + estcoef_this / Config.sig_freq)
    beta2 = 2 * xp.pi * (xp.polyval(coeff, pidx) - estbw / 2) - tstart * 2 * beta1  # 2ax+b=differential b=differential - 2 * beta1 * time
    coef2d_est = xp.array([to_scalar(beta1), to_scalar(beta2), 0])

    sig2 = reader.get(x1, x2-x1) * xp.exp(-1j * xp.polyval(coef2d_est, tsymbr))
    data0 = myfft(sig2, n=Config.fft_n, plan=Config.plan)
    freq1 = xp.fft.fftshift(xp.fft.fftfreq(Config.fft_n, d=1 / Config.fs))[xp.argmax(xp.abs(data0))]
    freq, valnew = optimize_1dfreq_fast(sig2, tsymbr, freq1, Config.fs / Config.fft_n * 5) # valnew may be as low as 0.3, only half the power will be collected
    # freq = freq1 # todo !!!
    # assert valnew > 0.3, f"{freq=} {freq1=} {valnew=}"
    if freq < 0: freq += estbw
    codex = freq / estbw * 2 ** Config.sf
    code = around(codex)
    # print(f"{codex=} {code=}")

    tmid = tstart * (code / 2 ** Config.sf) + tend * (1 - code / 2 ** Config.sf)
    tmid = tmid.item()
    x3 = math.ceil(tmid * Config.fs)

    nsymbr1 = xp.arange(x1, x3)
    tsymbr1 = nsymbr1 / Config.fs
    nsymbr2 = xp.arange(x3, x2)
    tsymbr2 = nsymbr2 / Config.fs

    beta2 = (2 * xp.pi * (xp.polyval(coeff, pidx) + estbw * (code / 2 ** Config.sf - 0.5))
             - tstart * 2 * beta1)
    coef2d_est2 = xp.array([to_scalar(beta1), to_scalar(beta2), 0])
    coef2d_est2_2d = xp.polyval(coef2d_est2, tstart) - startphase
    coef2d_est2[2] -= coef2d_est2_2d

    beta2a = (2 * xp.pi * (xp.polyval(coeff, pidx) + estbw * (code / 2 ** Config.sf - 1.5))
              - tstart * 2 * beta1)
    coef2d_est2a = xp.array([to_scalar(beta1), to_scalar(beta2a), 0])
    coef2d_est2a_2d = xp.polyval(coef2d_est2a, tmid) - xp.polyval(coef2d_est2, tmid)
    coef2d_est2a[2] -= coef2d_est2a_2d

    res2 = reader.get(x1, x3-x1).dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr1))) / xp.sum(xp.abs(reader.get(x1, x3-x1)))
    res2a = reader.get(x3, x2-x3).dot(xp.exp(-1j * xp.polyval(coef2d_est2a, tsymbr2))) / xp.sum(xp.abs(reader.get(x3, x2-x3)))

    if not (xp.abs(res2).item() > 0.7 or code > 2 ** Config.sf * 0.7) or not (xp.abs(res2a).item() > 0.7 or code < 2 ** Config.sf * 0.2):
        pltfig1(tsymbr1, xp.angle(reader.get(x1, x3-x1) * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr1))), title=f"{pidx=} 1st angle {codex=} pow={xp.abs(res2).item()}").show()
        pltfig1(tsymbr2, xp.angle(reader.get(x3, x2-x3) * xp.exp(-1j * xp.polyval(coef2d_est2a, tsymbr2))), title=f"{pidx=} 2st angle {codex=} pow={xp.abs(res2a).item()}").show()

    assert xp.abs(res2).item() > 0.7 or code > 2 ** Config.sf * 0.7, f"{pidx=} {code=} 1st power {xp.abs(res2).item()}<0.7"
    assert xp.abs(res2a).item() > 0.7 or code < 2 ** Config.sf * 0.2, f"{pidx=} {code=} 2nd power {xp.abs(res2a).item()}<0.7"

    endphase = xp.polyval(coef2d_est2a, tend)
    ifreq1 = 2 * xp.pi * (xp.polyval(coeff, pidx) + estbw * (code / 2 ** Config.sf - 0.5))
    ifreq2 = 2 * xp.pi * (xp.polyval(coeff, pidx) + estbw * (code / 2 ** Config.sf - 1.5))

    # startphase = endphase
    # powers.append(xp.abs(res2).item())
    # powers.append(xp.abs(res2a).item())
    # codephase2.append(xp.angle(res2).item())
    # codephase2.append(xp.angle(res2a).item())
    # codephase.append(xp.angle(res2).item())
    # codephase.append(xp.angle(res2a).item())
    # coef2d_ests.append(coef2d_est2)
    # coef2d_ests.append(coef2d_est2a)
    codes[pidx] = code
    codexdiffs.append(abs(codex - code))
pltfig1(None, xp.unwrap(codexdiffs), title="codexdiffs").show()

### Construct curves for the 2 codes, still using only coeff_time and estcoef

- now compute curves based on the code from previous block

In [ ]:
coef_f = estcoefs[0]
coef_t = coeff_time3
codes[Config.preamble_len] = 8
codes[Config.preamble_len + 1] = 16 ## TODO
pidx_delta = 0.75
betai = Config.bw / ((2 ** Config.sf) / Config.bw) * xp.pi # frequency slope to phase 2d slope, *pi

for pidx in range(Config.total_len):
    if pidx >= Config.sfdend:
        tstart_delta = -2.5e-6
    else:
        tstart_delta = 0.0
    code = codes[pidx]
    if pidx >= Config.sfdend:
        cfo_start = xp.polyval(coef_f, pidx - pidx_delta)
    else:
        cfo_start = xp.polyval(coef_f, pidx)

    bw_start = Config.bw * (1 + cfo_start / Config.sig_freq)

    if pidx >= Config.sfdend:
        tstart = xp.polyval(coef_t, pidx - pidx_delta) 
    else:
        tstart = xp.polyval(coef_t, pidx)
    if pidx >= Config.sfdend - 1: 
        tend = xp.polyval(coef_t, pidx + 1 - pidx_delta) 
    else:
        tend = xp.polyval(coef_t, pidx + 1)

    if pidx >= Config.sfdpos and pidx < Config.sfdend:
        coef1_x2 = - betai * (1 + 2 * cfo_start / Config.sig_freq) # coef1_x2 = bw * pi
        coef1_x = 2 * xp.pi * (xp.polyval(coef_f, pidx) + bw_start * (code / Config.n_classes + 0.5)) - tstart * 2 * coef1_x2 # freq at tstart = polyval(coef_f, pidx) - bw_start/2 + bw_start * (code / 2^sf-0.5)
    else:
        coef1_x2 = betai * (1 + 2 * cfo_start / Config.sig_freq) # coef1_x2 = bw * pi
        coef1_x = 2 * xp.pi * (xp.polyval(coef_f, pidx) + bw_start * (code / Config.n_classes - 0.5)) - tstart * 2 * coef1_x2 # freq at tstart = polyval(coef_f, pidx) - bw_start/2 + bw_start * (code / 2^sf-0.5)
    coef1 = xp.array([to_scalar(coef1_x2), to_scalar(coef1_x), 0])
    coef1_const = xp.polyval(coef1, tstart + tstart_delta) - xp.polyval(coeffitlist[1, pidx - 1], tstart + tstart_delta)
    coef1[2] -= coef1_const
    coeffitlist[0, pidx] = coef1 # continuing the last phase in coeffitlist
    
    # 2nd part
    # freq at tstart = polyval(coef_f, pidx) - bw_start/2 + bw_start * (code / 2^sf-0.5) - bw_start
    if pidx in range(Config.preamble_len, Config.sfdpos) or pidx >= Config.sfdend:
        if pidx >= Config.sfdend:
            tjump = xp.polyval(coef_t, pidx + 1 - code / Config.n_classes - pidx_delta)
        else:
            tjump = xp.polyval(coef_t, pidx + 1 - code / Config.n_classes)
        cfo_jump = xp.polyval(coef_f, pidx + 1 - code / Config.n_classes)
        bw_jump = Config.bw * (1 + cfo_jump / Config.sig_freq)
        coef2_x2 = coef1_x2 
        coef2_x = coef1_x - 2 * xp.pi * bw_start # 2ax+b=differential b=differential - 2 * coef1_x2 * time
        coef2 = xp.array([to_scalar(coef1_x2), to_scalar(coef2_x), 0])

        assert tstart < tjump < tend, f"{pidx=} {tstart=} {tjump=} {tend=}"
        coef2_const = xp.polyval(coef2, tjump) - xp.polyval(coeffitlist[0, pidx], tjump)
        coef2[2] -= coef2_const
        coeffitlist[1, pidx] = coef2
    else:
        coeffitlist[1, pidx] = coeffitlist[0, pidx]
        # coeffitlist[1, pidx, 0] = 0
        # coeffitlist[1, pidx, 1] = 0
        # coeffitlist[1, pidx, 2] = xp.polyval(coeffitlist[0, pidx], tend) # keep continuous phase

    # print(pidx, xp.polyval(coeffitlist[0, pidx], tstart))
    # print(pidx, xp.polyval(coeffitlist[1, pidx], tend))


    if pidx < Config.sfdend:
        x1 = math.ceil(xp.polyval(coef_t, pidx) * Config.fs)
        x2 = math.ceil(xp.polyval(coef_t, pidx + 1) * Config.fs)
        x3 = math.ceil(xp.polyval(coef_t, pidx + (1 - code / 2 ** Config.sf)) * Config.fs)
    else:
        x1 = math.ceil(xp.polyval(coef_t, pidx - pidx_delta) * Config.fs)
        x2 = math.ceil(xp.polyval(coef_t, pidx + 1 - pidx_delta) * Config.fs)
        x3 = math.ceil(xp.polyval(coef_t, pidx + (1 - code / 2 ** Config.sf) - pidx_delta) * Config.fs)
    if pidx == Config.sfdend - 1:
        x1 = math.ceil(xp.polyval(coef_t, pidx) * Config.fs)
        x2 = math.ceil(xp.polyval(coef_t, pidx + 1 - pidx_delta) * Config.fs)
        x3 = x2
        
    nsymbr1 = xp.arange(x1, x3)
    tsymbr1 = nsymbr1 / Config.fs
    sig1 = reader.get(x1, x3 - x1)
    res1 = sig1.dot(xp.exp(-1j * xp.polyval(coef1, tsymbr1)) )

    codephase[pidx] = xp.angle(res1)
    powers[pidx] = xp.abs(res1) / xp.sum(xp.abs(sig1))
    # print(f"{pidx=} 1st part {code=} {xp.angle(res1)=} pow={xp.abs(res1)/xp.sum(xp.abs(sig1))}")
    # if pidx in [0, 120, 239, 242, 243, 244]:
    if False:# pidx == Config.sfdend - 1:
        pltfig1(tsymbr1, xp.angle(sig1 * xp.exp(-1j * xp.polyval(coef1, tsymbr1))), title=f"residue {pidx=}").show()

        pltfig((
            (tsymbr1, xp.unwrap(xp.angle(sig1))), 
            (tsymbr1, xp.polyval(coef1, tsymbr1) - xp.polyval(coef1, tsymbr1[0]) + xp.angle(sig1[0])),
            ),
            title=f"preamble code {pidx=} {code=} fit curve coef1").show()


        pltfig1(tsymbr1, xp.polyval(coef1, tsymbr1) - xp.polyval(coef1, tsymbr1[0]) + xp.angle(sig1[0]) - xp.unwrap(xp.angle(sig1)),
                 title=f"preamble code {pidx=} {code=} fit curve coef1").show()

    
    if pidx in range(Config.preamble_len, Config.sfdpos) or pidx >= Config.sfdend:
        nsymbr2 = xp.arange(x3, x2)
        tsymbr2 = nsymbr2 / Config.fs
        sig2 = reader.get(x3, x2 - x3)
        res2 = sig2.dot(xp.exp(-1j * xp.polyval(coef2, tsymbr2)) )
        codephase_secondary[pidx] = wrap(xp.angle(res1) - xp.angle(res2))
        powers[pidx] = (xp.abs(res1) + xp.abs(res2)) / (xp.sum(xp.abs(sig1)) + xp.sum(xp.abs(sig2)))

        tsymbrA = xp.arange(x1, x2) / Config.fs
        sigA = reader.get(x1, x2 - x1)
        # print(f"{pidx=} 2nd part {code=} {xp.angle(res2)=} pow={xp.abs(res2)/xp.sum(xp.abs(sig2))}")

        # if pidx in [240, 241] or pidx in range(Config.sfdend, Config.sfdend + 2):
        if False:#pidx in range(Config.sfdend, Config.sfdend + 3):
            pltfig1(tsymbr1, xp.angle(sig1 * xp.exp(-1j * xp.polyval(coef1, tsymbr1))), title=f"residue {pidx=}").show()
            pltfig(((tsymbr1, xp.angle(sig1 * xp.exp(-1j * xp.polyval(coef1, tsymbr1)))),
                (tsymbr2, xp.angle(sig2 * xp.exp(-1j * xp.polyval(coef2, tsymbr2))))), title=f"residue {pidx=}").show()

            pltfig((
                (tsymbrA, xp.unwrap(xp.angle(sigA))), 
                (tsymbr1, xp.polyval(coef1, tsymbr1) - xp.polyval(coef1, tsymbr1[0]) + xp.angle(sig1[0])),
                (tsymbr2, xp.polyval(coef2, tsymbr2) - xp.polyval(coef1, tsymbr1[0]) + xp.angle(sig1[0])),
                ),
                title=f"preamble code {pidx=} {code=} fit curve coef1").show()


            # pltfig((
            #     (tsymbr1, xp.polyval(coef1, tsymbr1) - xp.polyval(coef1, tsymbr1[0]) + xp.angle(sig1[0]) - xp.unwrap(xp.angle(sigA))[:len(tsymbr1)]),
            #     (tsymbr2, xp.polyval(coef2, tsymbr2) - xp.polyval(coef2, tjump) + xp.polyval(coef1, tjump) - xp.polyval(coef1, tsymbr1[0]) + xp.angle(sig1[0]) - xp.unwrap(xp.angle(sigA))[len(tsymbr1):]),
            #     ),
            #     title=f"preamble code {pidx=} {code=} fit curve coef1").show()
pltfig(((xp.arange(Config.total_len), xp.unwrap(codephase[:Config.total_len])), (xp.arange(Config.preamble_len, Config.total_len), xp.unwrap(codephase_secondary[Config.preamble_len : Config.total_len]))), title="preamble+data unwrap phase").show()
pltfig1(xp.arange(Config.total_len ), powers[:Config.total_len ], title="preamble+data powers").show()
pltfig1(None, codes, title="preamble+data codes").show()


In [ ]:
pidx_delta = 0.75
coefs_fix = coeffitlist.copy()
post_int_x1 = []
post_int_y1 = []
post_int_x2 = []n
post_int_y2 = []
time_delta = 2.5e-6 # 2.5us, to avoid the noise at the beginning of the symbol
fix_angles1 = []
fix_angles2 = []
for pidx in range(Config.sfdend, Config.total_len):
    if pidx==282 or pidx==283: # skip these two symbols, they are not used in the payload
        print(f"debug {pidx=}, {codes[pidx]=}, skipping")
        continue
    code = codes[pidx]
    cfo_start = xp.polyval(coef_f, pidx - pidx_delta)
    bw_start = Config.bw * (1 + cfo_start / Config.sig_freq)

    tstart = xp.polyval(coef_t, pidx - pidx_delta) - time_delta
    tend = xp.polyval(coef_t, pidx + 1 - pidx_delta) - time_delta # last symbol is shorter
    tjump = xp.polyval(coef_t, pidx + 1 - code / Config.n_classes - pidx_delta)

    x1 = math.ceil(xp.polyval(coef_t, pidx - pidx_delta) * Config.fs)
    x2 = math.ceil(xp.polyval(coef_t, pidx + 1 - pidx_delta) * Config.fs)
    x3 = math.ceil((xp.polyval(coef_t, pidx + (1 - code / 2 ** Config.sf) - pidx_delta)) * Config.fs) ### DEBUG!!!TODO

    coefs_fix[0, pidx, 2] -= xp.polyval(coefs_fix[0, pidx] , tstart) - xp.polyval(coefs_fix[1, pidx - 1], tstart)
    coefs_fix[1, pidx, 2] -= xp.polyval(coefs_fix[1, pidx] , tjump) - xp.polyval(coefs_fix[0, pidx], tjump)

    fix_angle1 = xp.angle(reader.get(x1, x3 - x1).dot(xp.exp(-1j * xp.polyval(coefs_fix[0, pidx], xp.arange(x1, x3) / Config.fs))))
    fix_angle2 = xp.angle(reader.get(x3, x2 - x3).dot(xp.exp(-1j * xp.polyval(coefs_fix[1, pidx], xp.arange(x3, x2) / Config.fs))))
    coefs_fix[0, pidx, 2] += fix_angle1
    coefs_fix[1, pidx, 2] += fix_angle2
    fix_angles1.append(to_scalar(fix_angle1))
    fix_angles2.append(to_scalar(fix_angle2))
    # print(xp.angle(reader.get(x1, x3 - x1).dot(xp.exp(-1j * xp.polyval(coefs_fix[0, pidx], xp.arange(x1, x3) / Config.fs)))))
    # print(xp.angle(reader.get(x3, x2 - x3).dot(xp.exp(-1j * xp.polyval(coefs_fix[1, pidx], xp.arange(x3, x2) / Config.fs)))))

    # if pidx in range(Config.sfdend + 1, Config.sfdend + 3):
#     print(pidx)
#     sel1 = find_intersections(coefs_fix[1, pidx - 1], coefs_fix[0, pidx], tstart, reader, 1e-5, draw=(pidx if pidx in range(Config.sfdend + 1, Config.sfdend + 3) else None), remove_range = False)
#     if sel1 is not None:
#         post_int_x1.append(pidx)
#         post_int_y1.append(to_scalar(sel1 - tstart))
#         # print(f"post int 1 found at pidx={pidx} t={sel1 - tstart}")
#     sel2 = find_intersections(coefs_fix[0, pidx], coefs_fix[1, pidx], tjump, reader, 1e-5, draw=(pidx if pidx in range(Config.sfdend + 1, Config.sfdend + 3) else None), remove_range = False)
#     if sel2 is not None:
#         post_int_x2.append(pidx)
#         post_int_y2.append(to_scalar(sel2 - tjump))
#         # print(f"post int 2 found at pidx={pidx} t={sel2 - tjump}")
# pltfig1(post_int_x1, post_int_y1, title="time intersection tstart").show()
# pltfig1(post_int_x2, post_int_y2, title="time intersection tjump").show()
pltfig1(None, fix_angles1, title="angle tstart-2.5e-6").show()
pltfig1(None, fix_angles2, title="angle tjump-2.5e-6").show()


# Fit 243 244 245 Working


In [ ]:

# for pidx in range(Config.preamble_len + 2, Config.preamble_len + 5):
#     x1 = math.ceil(xp.polyval(coeff_time, pidx) * Config.fs)
#     x2 = math.ceil(xp.polyval(coeff_time, pidx + 1) * Config.fs)
#     if pidx == Config.preamble_len + 4:
#         x2 = math.ceil(xp.polyval(coeff_time, pidx + 0.25) * Config.fs)
#     sig = reader.get(x1, x2 - x1)
#     nsymbr = xp.arange(x1, x2)
#     tsymbr = nsymbr / Config.fs

#     estcoef_this = xp.polyval(estcoef, pidx)
#     beta1 = - betai * (1 + 2 * estcoef_this / Config.sig_freq)
#     estbw = Config.bw * (1 + estcoef_this / Config.sig_freq)
#     # print(f"EEE! {Config.bw * (estcoef_this / Config.sig_freq)=}")
#     beta2 = 2 * xp.pi * (estcoef_this + estbw / 2) - xp.polyval(coeff_time, pidx) * 2 * beta1 # 2ax+b=differential b=differential - 2 * beta1 * time
#     coef2d_est2 = xp.array([to_scalar(beta1), to_scalar(beta2), 0])

#     tstart = xp.polyval(coeff_time, pidx)
#     tend = xp.polyval(coeff_time, pidx + 1)
#     if pidx == Config.preamble_len + 4:
#         tend = xp.polyval(coeff_time, pidx + 0.25)
#     coef2d_est2_2d = xp.polyval(coef2d_est2, tstart) - xp.polyval(coeffitlist[1, pidx - 1], tstart)
#     coef2d_est2[2] -= coef2d_est2_2d
#     coeffitlist[0, pidx] = coef2d_est2
#     coeffitlist[1, pidx, -1] = coef2d_est2[-1] # share the same for continuous phase
#     res2 = sig.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr)))
#     fig = pltfig1(tsymbr, xp.angle(sig * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))), title=f"residue {pidx=}", fig=fig)
#     codephase[pidx] = xp.angle(res2)
#     powers[pidx] = xp.abs(res2) / xp.sum(xp.abs(sig))
#     print(f"Fit SFD symbol {pidx=} angle={xp.angle(res2)} pow={xp.abs(res2)/xp.sum(xp.abs(sig))}")
#     # pltfig1(None, xp.angle(sig * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))), title=f"residue {pidx=}").show()
#     # freq, power = optimize_1dfreq(sig * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr)), tsymbr, 0)
#     # print(f"EEE! {freq=} {power=}")

#     # pltfig(((tsymbr1, xp.angle(sig21 * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr1)))), (tsymbr2, xp.angle(sig22 * xp.exp(-1j * xp.polyval(coef2d_est22, tsymbr2))))), title=f"residue {pidx=}").show()

#     # pltfig((
#     #     (tsymbr, xp.unwrap(xp.angle(sig))), 
#     #     (tsymbr1, xp.polyval(coef2d_est2, tsymbr1) - xp.polyval(coef2d_est2, tsymbr1[0]) + xp.angle(sig21[0])),
#     #     (tsymbr2, xp.polyval(coef2d_est22, tsymbr2) - xp.polyval(coef2d_est22, tjump) + xp.polyval(coef2d_est2, tjump) - xp.polyval(coef2d_est2, tsymbr1[0]) + xp.angle(sig21[0])),
#     #     ),
#     #     title=f"preamble code {pidx=} fit curve coef2d_est2").show()
#     pltfig((
#         (tsymbr, xp.unwrap(xp.angle(sig))), 
#         (tsymbr, xp.polyval(coef2d_est2, tsymbr) - xp.polyval(coef2d_est2, tsymbr[0]) + xp.angle(sig[0])),
#         ),
#         title=f"SFD code {pidx=} fit curve coef2d_est2").show()
#     pltfig1( tsymbr, xp.unwrap(xp.angle(sig)) - (xp.polyval(coef2d_est2, tsymbr) - xp.polyval(coef2d_est2, tsymbr[0]) + xp.angle(sig[0])), title=f"SFD code {pidx=} fit diff").show()
# pltfig1(None, xp.unwrap(codephase), title="all unwrap phase").show()
# pltfig1(None, powers, title="all powers").show()

In [ ]:

# for pidx in range(Config.preamble_len + 4, Config.preamble_len + 5):
#     x1 = math.ceil(xp.polyval(coeff_time, pidx) * Config.fs)
#     x2 = math.ceil(xp.polyval(coeff_time, pidx + 0.25) * Config.fs)
#     nsymbr = xp.arange(x1, x2)
#     tsymbr = nsymbr / Config.fs
#     sig = reader.get(x1, x2 - x1)

#     estcoef_this = xp.polyval(estcoef, pidx)
#     beta1 = - betai * (1 + 2 * estcoef_this / Config.sig_freq)
#     estbw = Config.bw * (1 + estcoef_this / Config.sig_freq)
#     beta2 = 2 * xp.pi * (xp.polyval(estcoef, pidx) + estbw / 2) - xp.polyval(coeff_time, pidx) * 2 * beta1 # 2ax+b=differential b=differential - 2 * beta1 * time
#     coef2d_est2 = xp.array([to_scalar(beta1), to_scalar(beta2), 0])
#     coef2d_est2_2d = xp.polyval(coef2d_est2, xp.polyval(coeff_time, pidx)) - xp.polyval(
#         coeffitlist[pidx - 1], xp.polyval(coeff_time, pidx))
#     coef2d_est2[2] -= coef2d_est2_2d
#     cd2 = xp.angle(sig.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr)) ))#!!!!!!!!!!!! TODO here we align phase for the last symbol
#     print(f"WARN last phase {cd2=} manually add phase compensation")
#     # coef2d_est2[2] += cd2
#     # cd2 = xp.angle(sig.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr)) ))#!!!!!!!!!!!! TODO here we align phase for the last symbol
#     # assert abs(cd2) < 1e-4
#     coeffitlist[pidx] = coef2d_est2
#     res2 = sig.dot(xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr)))
#     codephase.append(xp.angle(res2).item())
#     powers.append(xp.abs(res2).item() / xp.sum(xp.abs(sig)).item())
#     fig=pltfig1(tsymbr, xp.angle(sig * xp.exp(-1j * xp.polyval(coef2d_est2, tsymbr))), title=f"residue {pidx=}", fig=fig)


In [ ]:

# anslist = []
# anslista = []
# anslistb = []
# anslist2 = []
# anslist2a = []
# anslist2b = []
# for pidx in range(2, len(codephase2), 2):
#     code = codes[pidx // 2]
#     tmid = tstart * (code / 2 ** Config.sf) + tend * (1 - code / 2 ** Config.sf)
#     tmid = tmid.item()
#     ifreq1 = xp.polyval(sqlist([2 * coef2d_ests[pidx][0], coef2d_ests[pidx][1]]), tstart ) - xp.polyval(sqlist([2 * coef2d_ests[pidx - 1][0], coef2d_ests[pidx - 1][1]]), tstart )
#     ifreq2 = xp.polyval(sqlist([2 * coef2d_ests[pidx + 1][0], coef2d_ests[pidx + 1][1]]), tmid ) - xp.polyval(sqlist([2 * coef2d_ests[pidx][0], coef2d_ests[pidx][1]]), tmid )
#     # print(pidx, ifreq1, ifreq2)
#     a1 = (wrap(codephase2[pidx] - codephase2[pidx - 1] - xp.pi) + xp.pi) / 2 / xp.pi / ifreq1
#     if ifreq1 < 0: a1 = (wrap(codephase2[pidx] - codephase2[pidx - 1] + xp.pi) - xp.pi) / 2 / xp.pi / ifreq1
#     assert a1>=0
#     a1a = a1 + 1 / abs(ifreq1)
#     a1b = a1 - 1 / abs(ifreq1)
#     anslist.append(to_scalar((a1)))
#     anslista.append(to_scalar((a1a)))
#     anslistb.append(to_scalar((a1b)))
#     a2 = wrap(codephase2[pidx + 1] - codephase2[pidx]) / 2 / xp.pi / ifreq2
#     a2a = a2 + 1 / abs(ifreq2)
#     a2b = a2 - 1 / abs(ifreq2)
#     anslist2.append(to_scalar((a2)))
#     anslist2a.append(to_scalar((a2a)))
#     anslist2b.append(to_scalar((a2b)))

# anslist = xp.unwrap(sqlist(anslist))
# tdifflist = anslist
# print(type(tdifflist))
# fig = pltfig1(None, tdifflist)
# fig = pltfig1(None, anslista, title="tdifflist", fig=fig)
# pltfig1(None, anslistb, title="tdifflista", fig=fig).show()
# anslist2 = xp.unwrap(sqlist(anslist2))
# tdifflist2 = anslist2
# fig = pltfig1(None, tdifflist2, title="tdifflist")
# fig = pltfig1(None, anslist2a, title="tdifflist", fig=fig)
# pltfig1(None, anslist2b, title="tdifflista", fig=fig).show()


# pltfig1(None, powers, title="powers").show()
# pltfig1(None, xp.unwrap(codephase), title="unwrap phase").show()


